In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path

cwd = Path.cwd()
ROOT = next(p for p in [cwd, *cwd.parents] if (p / ".git").exists())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.metrics import root_mean_squared_error, r2_score, roc_auc_score

from src.features.preprocessing import load_model_data
from src.evaluation import splits, leakage
from src.evaluation.make_splits import load_split, SPLITS

SPLIT_LIST = list(SPLITS)

In [ ]:
# ===== Settings: change these, then Restart & Run All =====
SPLIT = "10km"            # "10km", "10km_subbasin", "7.5km_subbasin", "5km_subbasin"
USE_SUBBASIN = False      # True: subbasin features replace elevation
MODEL_SEED = 0
THRESHOLD_MG_L = 10

In [ ]:
X, y, meta = load_model_data(subbasin=USE_SUBBASIN)
FEATURES = list(X.columns)

data = load_split(pd.concat([meta, X], axis=1).assign(log_nitrate=y), split=SPLIT)
print(f"split={SPLIT} | subbasin features={USE_SUBBASIN} | {len(FEATURES)} features")
print(data["fold"].value_counts().sort_index().to_dict(), "| holdout rows:", int(data["holdout"].sum()))


{0: 1035, 1: 972, 2: 1143, 3: 1041, 4: 1089} | holdout rows: 1089


In [8]:
# Set the training folds and the holdout

train_val = data[~data["holdout"]].reset_index(drop=True)       # Training set with 4 folds


X_tv = train_val[FEATURES]
y_tv = train_val["log_nitrate"]

# Print the sizes of the training folds
print(train_val["fold"].value_counts().sort_index().to_dict())

{0: 1035, 1: 972, 2: 1143, 3: 1041}


Here you can begin training your model.

In [9]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import root_mean_squared_error

In [ ]:
def make_model():
    """Replace with your model. A new, untrained model is created for every fold."""
    return DummyRegressor(strategy="mean")

xy = leakage.projected_xy(train_val, "gm_latitude", "gm_longitude")
oof_pred = np.full(len(train_val), np.nan)        # model prediction, out of fold
mean_pred = np.full(len(train_val), np.nan)       # baseline: training-fold mean
nearest_m = np.full(len(train_val), np.nan)       # distance to nearest training well
fold_scores = []

for train, test in splits.iter_folds(train_val["fold"]):
    model = make_model()
    model.fit(X_tv[train], y_tv[train])
    oof_pred[test] = model.predict(X_tv[test])
    mean_pred[test] = y_tv[train].mean()
    nearest_m[test] = leakage.nearest_training_distance(xy, train, test)

    fold_scores.append({
        "fold": int(train_val.loc[test, "fold"].iloc[0]),
        "n": int(test.sum()),
        "rmse_log": root_mean_squared_error(y_tv[test], oof_pred[test]),
        "rmse_mean": root_mean_squared_error(y_tv[test], mean_pred[test]),
    })

exceeds = np.expm1(y_tv) > THRESHOLD_MG_L
print(pd.DataFrame(fold_scores).round(3).to_string(index=False))
print(f"Overall RMSE (log): {root_mean_squared_error(y_tv, oof_pred):.3f} "
      f"| R²: {r2_score(y_tv, oof_pred):.3f} "
      f"| AUC (>{THRESHOLD_MG_L} mg/L): {roc_auc_score(exceeds, oof_pred):.3f}")


 fold  rmse_log
    0     1.227
    1     1.183
    2     1.222
    3     1.314
Overall RMSE (log): 1.238
Accuracy of the >10 mg/L call: 0.662


# Nested CV Example

In [ ]:
from src.models import random_forest
# "data" comes from load_split (see earlier template)
# this function creates the subfolds

development, X_dev, y_dev = random_forest.make_inner_subfolds(data, FEATURES, N_SUBFOLDS = 4, THRESHOLD_MG_L = 10, SEED = 77)

In [ ]:
#set the dictionary of tested hyperparameters
param_grid = {
"n_estimators": [10, 50, 100, 300, 500],
"max_features": [1.0, "sqrt", "log2", 0.5]
}

In [ ]:
# customize this function to your estimator. Leave all other fields as is. 
from sklearn.model_selection import GridSearchCV # type: ignore

def make_search_NAME(inner_cv, param_grid, SEED):
    return GridSearchCV(
        estimator=RandomForestRegressor(
            random_state=SEED,
            n_jobs=1,
        ),
        param_grid=param_grid,
        scoring="neg_root_mean_squared_error",
        cv=inner_cv,
        refit=True,
        n_jobs=-1,
    )

In [ ]:
#call the function fit_model_with_nested_cv

outer_results, best_params_by_fold = (
random_forest.fit_model_with_nested_cv(
        development,
        X_dev,
        y_dev,
        param_grid,
        make_search_NAME,
        N_SUBFOLDS=4,
        THRESHOLD_MG_L=10,
        SEED=99,
        PRINT=True
    )
)

In [ ]:
# Normal CV function (just outer folds)

model = RandomForestRegressor(n_estimators=100, max_features='log2', random_state=77, n_jobs=1)
random_forest.fit_model_with_normal_cv(development, X_dev, y_dev, model, THRESHOLD_MG_L = 10)

Here is a function that loops over all splits and evaluates model performence on each.

In [ ]:
def cv_summary(split, use_subbasin):
    X, y, meta = load_model_data(subbasin=use_subbasin)
    d = load_split(pd.concat([meta, X], axis=1).assign(log_nitrate=y), split=split)
    tv = d[~d["holdout"]].reset_index(drop=True)
    feats, yy = list(X.columns), tv["log_nitrate"]
    pred, base = np.full(len(tv), np.nan), np.full(len(tv), np.nan)
    for train, test in splits.iter_folds(tv["fold"]):
        pred[test] = make_model().fit(tv.loc[train, feats], yy[train]).predict(tv.loc[test, feats])
        base[test] = yy[train].mean()
    return {"split": split, "subbasin": use_subbasin,
            "rmse_log": root_mean_squared_error(yy, pred), "rmse_mean": root_mean_squared_error(yy, base),
            "r2": r2_score(yy, pred), "auc": roc_auc_score(np.expm1(yy) > THRESHOLD_MG_L, pred)}

pd.DataFrame([cv_summary(s, sb) for s in SPLIT_LIST
                                for sb in [False, True]]).round(3)


# Final Evaluation

Save for when we have chosen our final model (along with any hyperparameters) to evaluate on the test set.

In [ ]:
# ============================================================
# FINAL EVALUATION ON THE HOLD-OUT FOLD
# Run this ONCE, for the final chosen model only, after all
# model choices (features, hyperparameters) are finished.
# ============================================================
RUN_FINAL_EVALUATION = False

if RUN_FINAL_EVALUATION:
    holdout_df = data[data["holdout"]].reset_index(drop=True)
    final_model = make_model()
    final_model.fit(X_tv, y_tv)                                 # all four development folds
    holdout_pred = final_model.predict(holdout_df[FEATURES])
    y_ho = holdout_df["log_nitrate"]
    print(f"Hold-out ({SPLIT}) RMSE (log): {root_mean_squared_error(y_ho, holdout_pred):.3f} "
          f"| R²: {r2_score(y_ho, holdout_pred):.3f} "
          f"| AUC: {roc_auc_score(np.expm1(y_ho) > THRESHOLD_MG_L, holdout_pred):.3f}")
